## Example 1: Basic JSON serialization and deserialization

In [ ]:
import orjson
import json
import time

# Example 1: Basic JSON serialization and deserialization
print("Example 1 - Basic JSON serialization and deserialization")

try:
    print("orjson module imported successfully")
    
    # Test basic data types
    test_data = {
        "string": "Hello, World!",
        "integer": 42,
        "float": 3.14159,
        "boolean": True,
        "null": None,
        "array": [1, 2, 3, "four", 5.0],
        "nested": {
            "level1": {
                "level2": "deep value"
            }
        }
    }
    
    print(f"Original data: {len(str(test_data))} characters")
    
    # Serialize with orjson
    orjson_bytes = orjson.dumps(test_data)
    print(f"orjson serialized: {len(orjson_bytes)} bytes")
    print(f"orjson type: {type(orjson_bytes)}")
    
    # Compare with standard json
    std_json = json.dumps(test_data)
    print(f"std json serialized: {len(std_json)} characters")
    
    # Deserialize with orjson
    orjson_loaded = orjson.loads(orjson_bytes)
    print(f"orjson deserialized successfully")
    
    # Verify data integrity
    data_matches = orjson_loaded == test_data
    print(f"Data integrity check: {data_matches}")
    
    # Test specific values
    print(f"String value: {orjson_loaded['string']}")
    print(f"Nested value: {orjson_loaded['nested']['level1']['level2']}")
    print(f"Array length: {len(orjson_loaded['array'])}")
    
    # Test bytes to string conversion
    json_str = orjson_bytes.decode('utf-8')
    print(f"Decoded JSON string length: {len(json_str)}")
    
    print("Example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"orjson example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Advanced data types and serialization options

In [ ]:
import orjson
import datetime
import uuid
from dataclasses import dataclass
from decimal import Decimal

# Example 2: Advanced data types and serialization options
print("Example 2 - Advanced data types and serialization options")

try:
    # Define a dataclass for testing
    @dataclass
    class Person:
        name: str
        age: int
        email: str
    
    # Create test data with advanced types
    advanced_data = {
        "datetime": datetime.datetime(2023, 12, 25, 15, 30, 45),
        "date": datetime.date(2023, 12, 25),
        "uuid": uuid.UUID('550e8400-e29b-41d4-a716-446655440000'),
        "decimal": Decimal('123.456'),
        "person": Person("Alice", 30, "alice@example.com"),
        "bytes": b"binary data",
        "set": {1, 2, 3, 4, 5}
    }
    
    print(f"Advanced data types:")
    for key, value in advanced_data.items():
        print(f"  {key}: {type(value).__name__} = {value}")
    
    # Test serialization options
    print(f"\nTesting serialization options:")
    
    # Option 1: Default serialization — orjson has no native Decimal/bytes/set, so dumps() must raise
    # JSONEncodeError (a TypeError) at the first of them in insertion order: the Decimal (measured)
    try:
        default_json = orjson.dumps(advanced_data)
    except TypeError as e:
        assert str(e) == "Type is not JSON serializable: decimal.Decimal", e
        print(f"Default serialization: rejected Decimal as expected ({e})")
    else:
        raise AssertionError("orjson.dumps serialized Decimal/bytes/set with no default= hook")
    
    # Option 2: dataclass, datetime, date and UUID are NATIVE (OPT_SERIALIZE_DATACLASS is a no-op flag, value 0 —
    # measured); a default= hook is how the three unsupported types get through
    def to_json_default(obj):
        if isinstance(obj, Decimal):
            return str(obj)
        if isinstance(obj, bytes):
            return obj.decode()
        if isinstance(obj, set):
            return sorted(obj)
        raise TypeError
    dataclass_json = orjson.dumps(advanced_data, option=orjson.OPT_SERIALIZE_DATACLASS, default=to_json_default)
    assert orjson.loads(dataclass_json) == {
        "datetime": "2023-12-25T15:30:45", "date": "2023-12-25", "uuid": "550e8400-e29b-41d4-a716-446655440000",
        "decimal": "123.456", "person": {"name": "Alice", "age": 30, "email": "alice@example.com"},
        "bytes": "binary data", "set": [1, 2, 3, 4, 5],
    }, dataclass_json
    print("Dataclass serialization: SUCCESS (native dataclass/datetime/uuid + default= for Decimal/bytes/set)")
    
    # Test individual advanced types
    print(f"\nTesting individual types:")
    
    # Datetime serialization
    dt_data = {"timestamp": datetime.datetime.now()}
    dt_json = orjson.dumps(dt_data, option=orjson.OPT_SERIALIZE_DATACLASS)
    dt_loaded = orjson.loads(dt_json)
    print(f"Datetime: {dt_loaded['timestamp']}")
    
    # UUID serialization  
    uuid_data = {"id": uuid.uuid4()}
    uuid_json = orjson.dumps(uuid_data, option=orjson.OPT_SERIALIZE_DATACLASS)
    uuid_loaded = orjson.loads(uuid_json)
    print(f"UUID: {uuid_loaded['id']}")
    
    # Dataclass serialization
    person = Person("Bob", 25, "bob@test.com")
    person_json = orjson.dumps(person, option=orjson.OPT_SERIALIZE_DATACLASS)
    person_loaded = orjson.loads(person_json)
    print(f"Dataclass: {person_loaded}")
    
    # Test available options
    print(f"\nAvailable orjson options:")
    options = [attr for attr in dir(orjson) if attr.startswith('OPT_')]
    for option in options:
        value = getattr(orjson, option)
        print(f"  {option}: {value}")
    
    print("Example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"orjson example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Performance comparison and benchmarking

In [ ]:
import orjson
import json
import time

try:
    import numpy as np
except ImportError:  # NumPy is optional; orjson has no dependencies
    np = None

# Example 3: Performance comparison and benchmarking
print("Example 3 - Performance comparison and benchmarking")

try:
    # Create test data of varying sizes
    small_data = {"key": "value", "number": 42}
    
    medium_data = {
        "users": [
            {"id": i, "name": f"User {i}", "active": i % 2 == 0}
            for i in range(100)
        ],
        "metadata": {
            "version": "1.0",
            "created": "2023-12-25",
            "settings": {"debug": False, "timeout": 30}
        }
    }
    
    large_data = {
        "items": [
            {
                "id": f"item_{i}",
                "properties": {
                    "value": i * 1.5,
                    "tags": [f"tag_{j}" for j in range(5)],
                    "metadata": {"created": f"2023-12-{(i % 30) + 1:02d}"}
                }
            }
            for i in range(1000)
        ]
    }
    
    datasets = [
        ("small", small_data),
        ("medium", medium_data),
        ("large", large_data)
    ]
    
    print("Performance comparison (orjson vs standard json):")
    
    for name, data in datasets:
        print(f"\n{name.upper()} dataset:")
        
        # Measure orjson serialization
        start_time = time.time()
        for _ in range(100):  # Multiple iterations for timing
            orjson_result = orjson.dumps(data)
        orjson_serialize_time = time.time() - start_time
        
        # Measure standard json serialization
        start_time = time.time()
        for _ in range(100):
            json_result = json.dumps(data)
        json_serialize_time = time.time() - start_time
        
        # Measure orjson deserialization
        start_time = time.time()
        for _ in range(100):
            orjson.loads(orjson_result)
        orjson_deserialize_time = time.time() - start_time
        
        # Measure standard json deserialization
        start_time = time.time()
        for _ in range(100):
            json.loads(json_result)
        json_deserialize_time = time.time() - start_time
        
        # Calculate speedup
        serialize_speedup = json_serialize_time / orjson_serialize_time
        deserialize_speedup = json_deserialize_time / orjson_deserialize_time
        
        print(f"  Serialization:")
        print(f"    orjson: {orjson_serialize_time:.4f}s")
        print(f"    json: {json_serialize_time:.4f}s") 
        print(f"    Speedup: {serialize_speedup:.2f}x")
        
        print(f"  Deserialization:")
        print(f"    orjson: {orjson_deserialize_time:.4f}s")
        print(f"    json: {json_deserialize_time:.4f}s")
        print(f"    Speedup: {deserialize_speedup:.2f}x")
        
        print(f"  Size comparison:")
        print(f"    orjson: {len(orjson_result)} bytes")
        print(f"    json: {len(json_result)} bytes")
        print(f"    Same size: {len(orjson_result) == len(json_result)}")
    
    # Test with NumPy arrays if NumPy is installed
    print(f"\nNumPy array handling:")
    if np is not None:
        numpy_data = {
            "array": np.array([1, 2, 3, 4, 5]),
            "matrix": np.array([[1, 2], [3, 4]]),
            "float_array": np.array([1.1, 2.2, 3.3])
        }
        numpy_converted = {
            "array": numpy_data["array"].tolist(),
            "matrix": numpy_data["matrix"].tolist(),
            "float_array": numpy_data["float_array"].tolist()
        }
        
        # Without OPT_SERIALIZE_NUMPY, orjson rejects ndarray values
        try:
            orjson.dumps(numpy_data)
        except TypeError:
            print("NumPy without OPT_SERIALIZE_NUMPY: TypeError, as documented")
        else:
            raise AssertionError("orjson.dumps accepted ndarrays without OPT_SERIALIZE_NUMPY")
        
        # With the option, arrays serialize natively and match the tolist() form
        numpy_json = orjson.dumps(numpy_data, option=orjson.OPT_SERIALIZE_NUMPY)
        assert orjson.loads(numpy_json) == numpy_converted
        print("NumPy with OPT_SERIALIZE_NUMPY: SUCCESS")
    else:
        print("NumPy not installed; skipping")
    
    print("Example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"orjson example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Serialization options and formatting

In [ ]:
import orjson
import json

# Example 4: Serialization options and formatting
print("Example 4 - Serialization options and formatting")

try:
    # Test data with various formatting needs
    test_data = {
        "name": "Test User",
        "scores": [95.5, 87.2, 92.8, 88.1],
        "metadata": {
            "unicode": "Hello 世界 🌍",
            "special_chars": "tab:\t newline:\n quote:\"",
            "numbers": {"pi": 3.141592653589793, "large": 1234567890}
        },
        "arrays": [
            [1, 2, 3],
            ["a", "b", "c"],
            [True, False, None]
        ]
    }
    
    print("Testing serialization options:")
    
    # Default serialization
    default_json = orjson.dumps(test_data)
    print(f"Default: {len(default_json)} bytes")
    print(f"Sample: {default_json[:100].decode('utf-8')}...")
    
    # Pretty printing (indented)
    pretty_json = orjson.dumps(test_data, option=orjson.OPT_INDENT_2)
    print(f"\nPretty (indent 2): {len(pretty_json)} bytes")
    print("Sample:")
    print(pretty_json[:200].decode('utf-8'))
    
    # Sorted keys
    sorted_json = orjson.dumps(test_data, option=orjson.OPT_SORT_KEYS)
    sorted_data = orjson.loads(sorted_json)
    print(f"\nSorted keys: {list(sorted_data.keys())}")
    
    # Combined options
    combined_options = orjson.OPT_INDENT_2 | orjson.OPT_SORT_KEYS
    combined_json = orjson.dumps(test_data, option=combined_options)
    print(f"\nCombined options: {len(combined_json)} bytes")
    
    # Test Unicode handling
    unicode_data = {
        "english": "Hello World",
        "chinese": "你好世界", 
        "japanese": "こんにちは世界",
        "emoji": "🚀🌟💫⭐",
        "symbols": "αβγδε∑∆∇∈∉"
    }
    
    print(f"\nUnicode handling:")
    unicode_json = orjson.dumps(unicode_data)
    unicode_loaded = orjson.loads(unicode_json)
    
    for key, value in unicode_loaded.items():
        print(f"  {key}: {value}")
    
    # Test precision and number handling
    number_data = {
        "small_float": 0.000001,
        "large_float": 1234567890.123456789,
        "scientific": 1.23e-10,
        "negative": -999.999,
        "integer": 9223372036854775807,  # Large integer
        "zero": 0
    }
    
    print(f"\nNumber precision:")
    number_json = orjson.dumps(number_data)
    number_loaded = orjson.loads(number_json)
    
    for key, value in number_loaded.items():
        original = number_data[key]
        matches = value == original
        print(f"  {key}: {value} (matches: {matches})")
    
    # Test available options
    print(f"\nAll available options:")
    option_attrs = [attr for attr in dir(orjson) if attr.startswith('OPT_')]
    for attr in option_attrs:
        value = getattr(orjson, attr)
        print(f"  {attr}: {value}")
    
    # Test error handling
    print(f"\nError handling:")
    
    # Invalid JSON string
    try:
        invalid_json = '{"incomplete": '
        orjson.loads(invalid_json)
    except orjson.JSONDecodeError as e:
        print(f"JSON decode error caught: {type(e).__name__}")
    except Exception as e:
        print(f"Other error caught: {type(e).__name__}")
        raise
    
    print("Example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"orjson example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Integration patterns and real-world usage

In [ ]:
import orjson
import json
from typing import Any, Dict, List

# Example 5: Integration patterns and real-world usage
print("Example 5 - Integration patterns and real-world usage")

try:
    print("Testing real-world integration patterns:")
    
    # API response simulation
    api_response = {
        "status": "success",
        "data": {
            "users": [
                {
                    "id": 1,
                    "username": "alice123",
                    "profile": {
                        "name": "Alice Johnson",
                        "email": "alice@example.com",
                        "preferences": {
                            "theme": "dark",
                            "notifications": True,
                            "language": "en"
                        }
                    },
                    "last_login": "2023-12-25T10:30:00Z"
                },
                {
                    "id": 2, 
                    "username": "bob456",
                    "profile": {
                        "name": "Bob Smith",
                        "email": "bob@example.com", 
                        "preferences": {
                            "theme": "light",
                            "notifications": False,
                            "language": "es"
                        }
                    },
                    "last_login": "2023-12-24T15:45:00Z"
                }
            ],
            "pagination": {
                "page": 1,
                "per_page": 10,
                "total": 2,
                "total_pages": 1
            }
        },
        "timestamp": "2023-12-25T12:00:00Z"
    }
    
    print("API Response serialization:")
    api_json = orjson.dumps(api_response, option=orjson.OPT_INDENT_2)
    print(f"  Size: {len(api_json)} bytes")
    
    # Verify round-trip
    api_loaded = orjson.loads(api_json)
    roundtrip_success = api_loaded == api_response
    print(f"  Round-trip successful: {roundtrip_success}")
    print(f"  Users count: {len(api_loaded['data']['users'])}")
    
    # Configuration file simulation
    config_data = {
        "app": {
            "name": "MyApp",
            "version": "1.2.3",
            "debug": False
        },
        "database": {
            "host": "localhost",
            "port": 5432,
            "name": "myapp_db",
            "pool_size": 20
        },
        "cache": {
            "type": "redis",
            "host": "localhost",
            "port": 6379,
            "ttl": 3600
        },
        "logging": {
            "level": "INFO",
            "handlers": ["console", "file"],
            "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        }
    }
    
    print(f"\nConfiguration handling:")
    config_json = orjson.dumps(config_data, option=orjson.OPT_SORT_KEYS | orjson.OPT_INDENT_2)
    config_loaded = orjson.loads(config_json)
    
    print(f"  App name: {config_loaded['app']['name']}")
    print(f"  Database port: {config_loaded['database']['port']}")
    print(f"  Debug mode: {config_loaded['app']['debug']}")
    
    # Streaming/chunked data simulation
    print(f"\nStreaming data handling:")
    
    chunk_data = []
    for i in range(5):
        chunk = {
            "chunk_id": i,
            "data": [f"item_{j}" for j in range(i * 10, (i + 1) * 10)],
            "timestamp": f"2023-12-25T12:0{i}:00Z"
        }
        chunk_data.append(chunk)
    
    # Serialize each chunk
    total_size = 0
    for chunk in chunk_data:
        chunk_json = orjson.dumps(chunk)
        total_size += len(chunk_json)
        
        # Verify chunk can be deserialized
        chunk_loaded = orjson.loads(chunk_json)
        print(f"  Chunk {chunk_loaded['chunk_id']}: {len(chunk_loaded['data'])} items")
    
    print(f"  Total streaming size: {total_size} bytes")
    
    # Error handling and validation
    print(f"\nError handling examples:")
    
    test_cases = [
        ('{"valid": "json"}', True),
        ('{"invalid": json}', False),  # Missing quotes
        ('{"trailing": "comma",}', False),  # Trailing comma
        ('[1, 2, 3]', True),  # Valid array
        ('null', True),  # Valid null
        ('undefined', False),  # Invalid literal
    ]
    
    for json_str, should_work in test_cases:
        if should_work:
            result = orjson.loads(json_str)
            assert result == json.loads(json_str), (json_str, result)
            print(f"  '{json_str[:20]}...': SUCCESS")
            continue
        try:
            result = orjson.loads(json_str)
        except orjson.JSONDecodeError as e:
            print(f"  '{json_str[:20]}...': EXPECTED FAILURE ({type(e).__name__})")
        else:
            raise AssertionError(f"invalid JSON {json_str!r} was accepted as {result!r}")
    
    # Memory efficiency test
    print(f"\nMemory efficiency:")
    
    # Create large nested structure
    nested_data = {"level_0": {}}
    current = nested_data["level_0"]
    
    for i in range(10):
        current[f"level_{i+1}"] = {
            "data": list(range(50)),
            "metadata": {"level": i+1, "size": 50}
        }
        if i < 9:
            current[f"level_{i+1}"]["nested"] = {}
            current = current[f"level_{i+1}"]["nested"]
    
    nested_json = orjson.dumps(nested_data)
    print(f"  Nested structure: {len(nested_json)} bytes")
    
    # Verify deep nesting works
    nested_loaded = orjson.loads(nested_json)
    deep_value = nested_loaded["level_0"]["level_1"]["metadata"]["level"]
    print(f"  Deep access successful: level = {deep_value}")
    
    print("Example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"orjson example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")